# MLIP Active-Learning Tutorial

> New to ALF? Start with the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

This tutorial shows how the ALF `MLIPModel` (a MACE machine-learned interatomic
potential) is used in an **offline (pool-based) active-learning loop**. Our use case: starting from
a pretrained foundation model and **finetuning** it into an accurate force field for a single
organic molecule, while spending as few expensive labels as possible.

### How this differs from the design tutorials

The protein-design tutorials *maximise a fitness*. Here we do the opposite kind of active
learning: we **minimise model error using as few expensive labels as possible**. Each label is an
expensive quantum-chemistry calculation (DFT). We work from a fixed pool of DFT-labelled aspirin
configurations and let the model decide which ones are worth "paying" to reveal — choosing the
configurations where its committee of models *disagrees most* (often the most informative ones,
though, as we'll see, not always).

### Experiment overview

1. Download a pretrained MACE organics model and a pool of DFT-labelled aspirin configurations.
2. Finetune a committee (ensemble) of models on a small seed set and use their disagreement to
   estimate prediction uncertainty.
3. Each round: score the remaining candidate pool, acquire the most uncertain configurations,
   reveal their DFT labels, and finetune again.
4. Compare an uncertainty-driven acquisition against a random baseline, and see why acquisition
   choice matters.

### Framework Components

1. **Dataset** (`AspirinDataset`, defined below): loads DFT-labelled aspirin configurations and
   splits them into seed-train / validation / test / **candidate pool**. The pool holds the
   unlabelled configurations active learning chooses from.
2. **Surrogate Model** ([`MLIPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/)) wrapped in an [`EnsembleWrapper`](https://instadeepai.github.io/alf/api/alf_tools/models/) committee: each member finetunes the pretrained MACE model; their disagreement is our uncertainty.
3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): serves the remaining candidate pool each round.
4. **Acquisition Function** (`MaxVariance`, defined below): selects the configurations where the committee disagrees most (highest prediction variance). `RandomAcquisition` is the baseline for comparison.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): handles the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) with the dataset as scorer): reveals the precomputed DFT energy (and forces) for an acquired configuration via `dataset.query`.
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the active-learning loop.

## Setup

These tutorials are written for **dev mode** — running from a local clone of the ALF repository.
The MLIP tutorial needs the `mlip` package (the MACE force field), which ALF exposes through the
optional `mlip` extra (mirrored as the `mlip` dependency group in `tutorials/pyproject.toml`). From
the `tutorials/` directory, sync that group so the extra is installed alongside the tutorial
dependencies:

```bash
uv sync --group mlip   # installs alf_core, alf_tools[mlip] and tutorial deps (CPU PyTorch by default)
```

Register the environment as a Jupyter kernel, then select the `alf` kernel in this notebook:

```bash
uv run ipython kernel install --user --env VIRTUAL_ENV "$(pwd)/.venv" --name=alf
```

`uv sync` installs the **CPU** build of PyTorch by default. For GPU acceleration, see the
[GPU support section of the Installation Guide](https://instadeepai.github.io/alf/installation.html#gpu-support).

**Not running from a clone?** If you installed ALF with `pip`, run the optional cell below to
install this tutorial's dependencies into the current kernel, then restart the kernel.

In [ ]:
# Optional — only needed if you are NOT running from a cloned repo via `uv sync`.
# Installs ALF and this tutorial's dependencies into the current kernel, then restart the kernel.
# %pip install "alf_tools[mlip]" matplotlib pandas huggingface_hub

### Step 0: Download the Model and Dataset

The cell below downloads the pretrained MACE organics model and the public **rMD17 aspirin** dataset
(both from Hugging Face, public, no credentials) into the local cache. The aspirin configurations
carry DFT energies and forces, so no live quantum-chemistry engine is needed.

In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download, snapshot_download

# Download the pretrained MACE organics foundation model and pass its zip path directly
# to MLIPModel. Keeping tutorial downloads under data/ means they stay out of git.
DATA_ROOT = Path("data")
models_dir = DATA_ROOT / "pretrained_models"
models_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="mace_organics_02.zip",
    local_dir=str(models_dir),
)
mace_model_path = models_dir / "mace_organics_02.zip"

# Public rMD17 aspirin dataset (DFT energies + forces) from the mlip tutorials collection.
# rMD17 ships disjoint train/val/test files; we use the train file for the seed/pool and
# the *separate* test file as a genuinely held-out test set (see the leakage note below).
DATA_DIR = DATA_ROOT / "aspirin"
snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="training/rmd17_aspirin_*",
    local_dir=str(DATA_DIR),
)
ASPIRIN_TRAIN_XYZ = DATA_DIR / "training" / "rmd17_aspirin_train.xyz"
ASPIRIN_TEST_XYZ = DATA_DIR / "training" / "rmd17_aspirin_test.xyz"

print(f"✅ Pretrained model present at {mace_model_path}")
print(f"✅ rMD17 aspirin train/test present at {ASPIRIN_TRAIN_XYZ.parent}")

### Step 1: Import Required Libraries

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from alf_core import (
    AcquisitionFunction,
    Candidate,
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    LabelledCandidates,
    Optimizer,
    Oracle,
    State,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig, SubsampleConfig
from alf_tools.models.mlip import MLIPModel, MLIPModelConfig, MLIPTrainConfig
from alf_tools.models.utils.mlip_utils import (
    load_extxyz_as_labelled_candidates,
)
from mlip.training import TrainingLoop
from mlip.training.optimizer_config import OptimizerConfig

print("✅ All imports successful!")

### Step 2: Load the rMD17 Aspirin Dataset

We load configurations of **aspirin** — sampled from molecular-dynamics trajectories in the revised
MD17 (rMD17) dataset. Each configuration carries a DFT energy (our regression label) and forces
(stored in each candidate's `features["forces"]` for MLIP training). We use
`load_extxyz_as_labelled_candidates`, which delegates extxyz parsing to `mlip`'s
`ExtxyzReader` and adapts the resulting `ChemicalSystem`s into ALF candidates.

> **⚠️ Avoiding train/test leakage.** rMD17 frames come from MD trajectories, so nearby frames are
> highly correlated — almost duplicates. A naive random split of one file would scatter
> near-identical structures across train *and* test, making the test error look far better than the
> model's true generalisation. To avoid this we keep the splits **disjoint by source**: the
> seed-train / validation / candidate-pool come from `rmd17_aspirin_train.xyz`, while the held-out
> **test set is drawn from the separate `rmd17_aspirin_test.xyz`**. (For your own MD data, split by
> trajectory or by time, never uniformly at random over frames.)

We subsample `N_POOL` configs from the train file (carved into seed-train / validation /
candidate-pool) and `N_TEST` configs from the test file. Active learning then reveals labels from
the pool a few configurations at a time.

In [ ]:
DATA_SEED = 51505
N_POOL = 150  # configs from the train trajectory -> seed-train + validation + candidate pool
N_TEST = 50  # configs from the *separate* test trajectory -> held-out test set


class AspirinDataset(BaseDataset):
    """Pre-labelled aspirin conformers, with a disjoint held-out test set.

    The train file is split by ratio into seed-train / validation / candidate-pool;
    the test split is replaced with `test_data` (drawn from a separate trajectory file)
    so temporally-correlated frames never leak between train and test.
    """

    def __init__(
        self,
        config: BaseDatasetConfig,
        train_data: LabelledCandidates,
        test_data: LabelledCandidates,
    ):
        """Store the train pool and the disjoint test set; `setup()` performs the split."""
        super().__init__(config)
        self._data = train_data
        self._test_data = test_data

    def load_dataset(self) -> LabelledCandidates:
        """Return the train-file configurations (split into seed-train / val / pool)."""
        return self._data

    def setup(self) -> None:
        """Split the train file, then swap in the disjoint test set from the test file."""
        super().setup()
        self.splits["test"] = self._test_data


def make_aspirin_dataset(
    train_data: LabelledCandidates | None = None,
    test_data: LabelledCandidates | None = None,
    seed: int = DATA_SEED,
) -> AspirinDataset:
    """Build and set up an `AspirinDataset` with the CPU-tiny split used in this tutorial.

    Pass `train_data`/`test_data` to reuse already-loaded configurations; otherwise they
    are loaded fresh from `ASPIRIN_TRAIN_XYZ` / `ASPIRIN_TEST_XYZ`.
    """
    if train_data is None:
        train_data = sample_labelled_candidates(
            load_extxyz_as_labelled_candidates(ASPIRIN_TRAIN_XYZ), N_POOL, seed
        )
    if test_data is None:
        test_data = sample_labelled_candidates(
            load_extxyz_as_labelled_candidates(ASPIRIN_TEST_XYZ), N_TEST, seed
        )
    ds = AspirinDataset(
        BaseDatasetConfig(
            name="rmd17_aspirin",
            modality=Modality.TABULAR,
            seed=seed,
            train_ratio=0.12,
            validation_frac=0.2,
            test_ratio=0.0,  # test comes from the separate file, not a ratio split
            split_type="random",
            problem_type=ProblemType.REGRESSION,
        ),
        train_data,
        test_data,
    )
    ds.setup()
    return ds


def sample_labelled_candidates(
    data: LabelledCandidates, n_configs: int, seed: int
) -> LabelledCandidates:
    """Return a reproducible subset of labelled candidates for this CPU-sized tutorial."""
    if len(data) <= n_configs:
        return data
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(data), size=n_configs, replace=False)
    candidates, labels = data[indices]
    return LabelledCandidates(candidates=candidates, labels=labels)


train_data = sample_labelled_candidates(
    load_extxyz_as_labelled_candidates(ASPIRIN_TRAIN_XYZ), N_POOL, DATA_SEED
)
test_data = sample_labelled_candidates(
    load_extxyz_as_labelled_candidates(ASPIRIN_TEST_XYZ), N_TEST, DATA_SEED
)


dataset = make_aspirin_dataset(train_data, test_data)
print(dataset)
print(
    f"✅ rMD17 aspirin dataset ready — pool={len(dataset.candidate_pool)}, "
    f"test={len(dataset.test_dataset)} (disjoint test trajectory)"
)

### Step 3: The Oracle (precomputed DFT)

The oracle is the expensive ground-truth evaluator. Here each aspirin configuration already has a
DFT energy, so the oracle simply **reveals** that label on demand: `Oracle(scorer=dataset)` calls
`dataset.query(...)` to look up the energy for an acquired configuration. This mirrors production
active learning, where labelling is costly and is therefore spent sparingly on the most informative
structures.

> **Offline vs online.** This is the **offline** setting: the oracle matches acquired candidates to
> their labels *by object identity* against the dataset, so it only works for configurations already
> in the pool. If instead you want to **generate new structures on the fly** and label them with a
> live evaluator (a running DFT/xTB calculation, or a pretrained MLIP used *as* the oracle), use a
> `BaseModel`-backed oracle and a generative search — see the
> [online design tutorial](https://github.com/instadeepai/alf/blob/main/tutorials/experiments/online_design_tutorial.ipynb).

In [ ]:
# Offline oracle: reveal the precomputed DFT label for an acquired configuration.
# In production this is an expensive DFT calculation; here the labels already exist in
# the dataset, so the oracle is a lookup (`dataset.query`) keyed by candidate identity.
oracle = Oracle(scorer=dataset)
print("✅ Oracle ready (offline lookup of precomputed DFT labels)!")

### Step 4: Search and Acquisition

`DatasetSearch` serves the remaining **candidate pool** — the aspirin configurations not yet
acquired. `MaxVariance` (used next) scores them by committee disagreement, picking the
configurations the ensemble is least sure about. `RandomAcquisition` is the baseline that ignores
the model and picks at random, so we can show that uncertainty-driven selection actually helps.

In [ ]:
class MaxVariance(AcquisitionFunction):
    """Select conformers where the committee disagrees most (highest prediction variance).

    This is query-by-committee uncertainty sampling: the structures the ensemble is least
    certain about are the most informative to label next. It is the natural acquisition for
    *reducing model error* (our goal), as opposed to maximising a property.
    """

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        """Score each candidate by its committee variance (higher = more uncertain)."""
        predictions = state.surrogate.predict(search_candidates)
        if predictions.variances is None:
            raise ValueError("MaxVariance requires committee variances; use an ensemble surrogate.")
        return LabelledCandidates(candidates=search_candidates, labels=predictions.variances)


class RandomAcquisition(AcquisitionFunction):
    """Baseline acquisition that scores candidates uniformly at random."""

    def __init__(self, seed: int = 0):
        """Seed the RNG used to score candidates."""
        self.seed = seed

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        """Assign each candidate a random acquisition value."""
        rng = np.random.default_rng(self.seed)
        scores = rng.random(len(search_candidates))
        return LabelledCandidates(candidates=search_candidates, labels=scores)


# Offline pool search: serve the remaining (unlabelled) candidate pool each round.
search_fn = DatasetSearch()
print("✅ Search strategy initialised (offline candidate pool)!")

### Step 5: The Surrogate — a Committee of Finetuned MACE Models

Our surrogate is an ensemble (committee) of `MLIPModel`s. Each member finetunes the **same**
pretrained MACE model, but on a different bootstrap resample of the training data, so the members
end up slightly different. Where they disagree most, the model is most uncertain — exactly the
conformers worth labelling. `EnsembleWrapper.predict` returns both the mean energy and the
variance across members.

> **A note on atomic reference energies (E0s).** MACE adds a per-element reference energy to
> learned residual energies. During finetuning, `MLIPModel` asks mlip's `MULTI` dataset builder
> to derive target-domain E0s from the current training split, merge them with the pretrained
> species table, and then collapse that target map back to the single-head model before training.
> That keeps the pretrained embedding and charge tables shape-compatible while aligning the
> absolute energy zero with rMD17. For a single molecule the target E0 solve is formally
> **degenerate**: every aspirin structure has the identical composition (9 C, 8 H, 4 O), so the
> individual per-element energies are not identifiable. Here that is harmless because the fitted
> *total* reference for the constant composition is what controls the absolute energy offset.

In [ ]:
def mlip_factory(seed: int) -> MLIPModel:
    """Build a finetuning MLIPModel (from the pretrained MACE model) with the given seed."""
    return MLIPModel(
        model_config=MLIPModelConfig(model_path=mace_model_path, model_type="mace"),
        train_config=MLIPTrainConfig(
            batch_size=2,
            optimizer_config=OptimizerConfig(
                init_learning_rate=1e-3,
                peak_learning_rate=1e-3,
                final_learning_rate=1e-3,
            ),
            training_loop_config=TrainingLoop.Config(num_epochs=8),
        ),
        seed=seed,
    )


def make_surrogate(n_members: int = 2) -> Surrogate:
    """Build a committee surrogate of finetuned MACE models (1 member = no uncertainty)."""
    return Surrogate(
        model=EnsembleWrapper(
            model_factory=mlip_factory,
            config=EnsembleWrapperConfig(
                base_seed=0,
                n_members=n_members,
                subsample=SubsampleConfig(fraction=1.0, replace=True),
            ),
        )
    )


surrogate = make_surrogate(n_members=2)
print("✅ Surrogate committee initialised (2 finetuned MACE members)!")

### Step 6: Optimizer and Design Task

`MaxVariance` ranks candidates purely by the committee's prediction variance, so each round we
label the pool configurations the ensemble disagrees on most — classic query-by-committee active
learning. The `Optimizer` combines it with the `DatasetSearch` over the candidate pool; `DesignTask`
runs the rounds.

In [ ]:
acquisition_fn = MaxVariance()
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

num_acq_rounds = 10
acq_batch_size = 5
task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)
print(f"✅ Optimizer + DesignTask ready ({num_acq_rounds} rounds × {acq_batch_size} labels)!")

### Step 7: Run the Active-Learning Experiment (uncertainty-driven)

Each round: take the remaining candidate pool, score it with the committee, acquire the most
uncertain configurations, reveal their DFT labels, finetune the committee, and evaluate on the
held-out test set.

> **⏱️ Slowest cell.** This is the most expensive step in the notebook: it finetunes a 2-member
> MACE committee once per acquisition round (plus the initial fit). On a laptop CPU expect a few
> minutes; it is much faster on GPU. The random baseline below is comparable in cost.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)  # keep MACE output quiet in the notebook

maxvar_path = Path("results/mlip_design/")
if maxvar_path.exists():
    shutil.rmtree(maxvar_path)
loggers = [TerminalStateLogger(), FileStateLogger(output_path=maxvar_path)]

state = task.setup(dataset=dataset, surrogate=surrogate)
print("🚀 Running uncertainty-driven active learning...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ Uncertainty-driven experiment completed!")

### Step 8: Random-Acquisition Baseline

To isolate the effect of the *acquisition strategy*, the baseline runs the identical loop on the
same aspirin splits with the **same 2-member committee surrogate** — the only thing that changes is
*how* configurations are chosen (uniformly at random instead of by committee variance). Keeping the
surrogate fixed makes the two learning curves a fair, apples-to-apples comparison.

> **⏱️ Slow cell** (comparable to Step 7): another committee finetuned across acquisition rounds.

In [ ]:
# Rebuild a fresh dataset so the baseline starts from the same initial splits.
dataset_random = make_aspirin_dataset()
random_oracle = Oracle(scorer=dataset_random)  # oracle is bound to THIS dataset's labels

# Same 2-member committee as the MaxVariance run — only the acquisition differs.
random_optimizer = Optimizer(acquisition_fn=RandomAcquisition(seed=0), search_fn=search_fn)
random_surrogate = make_surrogate(n_members=2)
random_task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

random_path = Path("results/mlip_design_random/")
if random_path.exists():
    shutil.rmtree(random_path)
random_loggers = [TerminalStateLogger(), FileStateLogger(output_path=random_path)]

random_state = random_task.setup(dataset=dataset_random, surrogate=random_surrogate)
print("🚀 Running random-acquisition baseline...")
random_task.run(
    random_state, state_loggers=random_loggers, optimizer=random_optimizer, oracle=random_oracle
)
print("✅ Baseline completed!")

### Step 9: Results

The headline metric is the **learning curve**: test-set energy error against the number of labels
acquired. Because both runs share the same committee surrogate, any difference is down to the
acquisition strategy alone. We then check absolute accuracy with an **energy parity plot**
on the held-out test set. Force labels are still used during MACE training, but `MLIPModel.predict()`
returns energy predictions through the standard ALF `Predictions` interface.

In [ ]:
maxvar_metrics = pd.read_csv("results/mlip_design/metrics.csv")
rnd_metrics = pd.read_csv("results/mlip_design_random/metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLIP Active Learning: uncertainty-driven vs random", fontsize=15, fontweight="bold")

for df, label, color in [
    (maxvar_metrics, "MaxVariance (committee)", "#e74c3c"),
    (rnd_metrics, "Random (committee)", "#3498db"),
]:
    axes[0].plot(
        df["dataset/num_train"], df["surrogate/test_mse"], marker="o", label=label, color=color
    )
    axes[1].plot(
        df["dataset/num_train"], df["surrogate/test_spearman"], marker="o", label=label, color=color
    )

axes[0].set_xlabel("Number of labelled structures")
axes[0].set_ylabel("Test energy MSE (eV²)")
axes[0].set_title("Learning curve (lower is better)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of labelled structures")
axes[1].set_ylabel("Test Spearman")
axes[1].set_title("Rank correlation (higher is better)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

**Reading the learning curves.** The left panel is the headline result: test-set energy MSE (eV²,
lower is better) on the held-out test *trajectory* against the number of labelled structures, which
grows by `acq_batch_size` each round. The right panel tracks rank correlation (Spearman, higher is
better) over the same budget. Both curves use the **same 2-member committee** — only the acquisition
differs — so any separation is attributable to the strategy, and the test set is a disjoint
trajectory so the numbers reflect genuine generalisation rather than leakage.

On a problem this small — 150 pool configs, a 2-member committee, and only a few rounds — expect the
two curves to be close and a little noisy: bootstrap resampling of a handful of structures gives
only a coarse uncertainty signal, so small gaps are not decisive. The point here is that the
end-to-end loop runs and the error trends downward as labels are spent; we return to *why*
max-variance need not win on a toy problem in the conclusion.

Next we check absolute energy accuracy on the held-out test trajectory.

In [ ]:
# Energy parity on the held-out test trajectory, using the random-baseline committee.
test_cands = dataset_random.test_dataset.candidates
test_true = dataset_random.test_dataset.labels
test_pred = random_surrogate.predict(test_cands).means  # committee-mean energy

energy_mae = float(np.mean(np.abs(test_pred - test_true)))
print(f"Test energy MAE: {energy_mae:.4f} eV")

fig, ax = plt.subplots(figsize=(6.5, 5.5))

ax.scatter(test_true, test_pred, alpha=0.7, color="#9b59b6")
elims = [min(test_true.min(), test_pred.min()), max(test_true.max(), test_pred.max())]
ax.plot(elims, elims, "k--", alpha=0.5)
ax.set_xlabel("DFT energy (eV)")
ax.set_ylabel("Predicted energy (eV)")
ax.set_title(f"Energy parity (MAE {energy_mae:.3f} eV)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Reading the parity plot.** Each point is a held-out configuration: its DFT energy (x) against
the finetuned model's prediction (y), with the dashed line marking perfect agreement. Energies span
only a narrow window (~1–2 eV about the mean), so a large constant vertical offset after E0
retargeting would signal remaining absolute-energy calibration error rather than poorly learned
energy *differences*. Force labels are included during MACE training via each candidate's
`features["forces"]`, but this ALF wrapper exposes energy predictions only.

## Conclusion

We built an offline (pool-based) active-learning loop for a machine-learned interatomic potential
with ALF: finetune a pretrained MACE committee on a few DFT-labelled aspirin configurations,
estimate uncertainty from committee disagreement, acquire more configurations from a candidate pool,
and repeat. Every component is a standard ALF abstraction — `AspirinDataset`, the dataset-lookup
`Oracle`, `DatasetSearch`, the `EnsembleWrapper` surrogate, `MaxVariance`/`RandomAcquisition`,
`Optimizer`, and `DesignTask` — and both acquisition strategies share the same committee surrogate
on a leakage-free (disjoint-trajectory) test set.

This example is deliberately tiny and fast, so treat the learning curves as illustrative rather
than conclusive. On a problem this small, uncertainty (max-variance) sampling does not reliably beat
random selection — a useful reminder that the acquisition strategy must be matched to the problem
and validated, not assumed. Query-by-committee active learning is the workhorse of *production* MLIP
training, where it selects from large, diverse pools of physically meaningful structures.

### Next steps

- **Scale up** so the uncertainty signal can show: more committee members, more acquisition rounds,
  a larger pool, and a larger held-out test set (rMD17 aspirin ships 1200/900/900 configs).
- **Try a stronger uncertainty estimate**: a deep-kernel / last-layer GP or a last-layer Laplace
  approximation on the MACE features gives calibrated epistemic variance (the latter is the
  NTK-flavoured, more principled cousin of an ensemble).
- **Go online**: generate candidates on the fly and label them with a live oracle (DFT/xTB, or a
  pretrained MLIP used *as* the oracle) — see the offline-vs-online note in Step 3.
- **Try a different molecule** or train from scratch (`model_path=None`) for chemistries with no
  compatible foundation model.

**Happy modelling!** ⚛️🔬✨

In [ ]:
for p in [Path("results/mlip_design/"), Path("results/mlip_design_random/")]:
    if p.exists():
        shutil.rmtree(p)
print("✅ Results directories cleaned up!")